In [0]:
%sql
-- DIM CUSTOMER (SCD TYPE 2 STRUCTURE)
CREATE OR REPLACE TABLE retail_lakehouse.gold.dim_customer
(
    CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID INT,
    CustomerName STRING,
    Email STRING,
    City STRING,
    Address STRING,
    StartDate DATE,
    EndDate DATE,
    IsActive BOOLEAN
)USING DELTA;

In [0]:
%sql
-- INITIAL LOAD INTO DIM CUSTOMER

INSERT INTO retail_lakehouse.gold.dim_customer
(
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
SELECT
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    CURRENT_DATE() AS StartDate,
    DATE '9999-12-31' AS EndDate,
    TRUE AS IsActive
FROM retail_lakehouse.silver.customers;

In [0]:
%sql
-- DIM PRODUCT
CREATE OR REPLACE TABLE retail_lakehouse.gold.dim_product AS
SELECT
    monotonically_increasing_id() AS ProductSK,
    ProductID,
    ProductName,
    Category,
    UnitPrice
FROM retail_lakehouse.silver.products;

In [0]:
%sql
-- DIM STORE
CREATE OR REPLACE TABLE retail_lakehouse.gold.dim_store AS
SELECT
    monotonically_increasing_id() AS StoreSK,
    StoreID,
    StoreName,
    Region
FROM retail_lakehouse.silver.stores;

In [0]:
%sql
-- FACT SALES
CREATE OR REPLACE TABLE retail_lakehouse.gold.fact_sales AS
SELECT
    monotonically_increasing_id() AS SalesSK,
    s.TransactionID,
    c.CustomerSK,
    p.ProductSK,
    st.StoreSK,
    s.Quantity,
    s.Quantity * p.UnitPrice AS Amount,
    s.TxnDate
FROM retail_lakehouse.silver.sales s
JOIN retail_lakehouse.gold.dim_customer c
    ON s.CustomerID = c.CustomerID
    AND c.IsActive = TRUE
JOIN retail_lakehouse.gold.dim_product p
    ON s.ProductID = p.ProductID
JOIN retail_lakehouse.gold.dim_store st
    ON s.StoreID = st.StoreID;

In [0]:
%sql
-- FACT TABLE VALIDATION
SELECT COUNT(*) AS fact_sales_count
FROM retail_lakehouse.gold.fact_sales;

SELECT *
FROM retail_lakehouse.gold.fact_sales
LIMIT 10;

In [0]:
%sql
-- REFERENTIAL INTEGRITY VALIDATION
-- CUSTOMER RI CHECK
SELECT *
FROM retail_lakehouse.silver.sales s
LEFT JOIN retail_lakehouse.gold.dim_customer c
ON s.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL;

-- PRODUCT RI CHECK
SELECT *
FROM retail_lakehouse.silver.sales s
LEFT JOIN retail_lakehouse.gold.dim_product p
ON s.ProductID = p.ProductID
WHERE p.ProductID IS NULL;

-- STORE RI CHECK
SELECT *
FROM retail_lakehouse.silver.sales s
LEFT JOIN retail_lakehouse.gold.dim_store st
ON s.StoreID = st.StoreID
WHERE st.StoreID IS NULL;

In [0]:
%sql
-- DERIVED AMOUNT VALIDATION
SELECT
    s.TransactionID,
    s.Quantity,
    p.UnitPrice,
    f.Amount
FROM retail_lakehouse.gold.fact_sales f

JOIN retail_lakehouse.silver.sales s
ON f.TransactionID = s.TransactionID

JOIN retail_lakehouse.silver.products p
ON s.ProductID = p.ProductID

LIMIT 10;


In [0]:
%sql
-- DATA TYPE VALIDATION
DESCRIBE retail_lakehouse.gold.dim_customer;

DESCRIBE retail_lakehouse.gold.dim_product;

DESCRIBE retail_lakehouse.gold.dim_store;

DESCRIBE retail_lakehouse.gold.fact_sales;